# FinOps | Medição e projeção mensal de custos (USD)

Duas partes:
1. **Medido**: duração e linhas de `monitoring.pipeline_runs` / `streaming_runs`.
2. **Projeção mensal** em `monitoring.finops_projecao_mensal`: serviço, unidade, quantidade, preço unitário e custo mensal.

**Os preços abaixo são PREMISSAS editáveis (widgets), não valores faturados.** Preencha com os preços da sua conta/região na data da
cotação e registre a fonte. A conta real (Databricks system tables / AWS Cost Explorer / BigQuery INFORMATION_SCHEMA) deve ser usada
para conferir a estimativa quando disponível.

In [0]:
dbutils.widgets.text("preco_dbu_usd", "[PREENCHER]", "Preço USD por DBU (serverless jobs) — PREENCHER")
dbutils.widgets.text("dbu_por_execucao", "[PREENCHER]", "DBU cobrados por execução COMPLETA do pipeline (system.billing.usage) — PREENCHER")
dbutils.widgets.text("dbu_por_hora", "[PREENCHER]", "(alternativa) DBU/hora — só usado se dbu_por_execucao estiver vazio")
dbutils.widgets.text("bq_gib_por_execucao", "", "(opcional) GiB lidos no BigQuery por carga completa (dry run) — se vazio, usa finops_ingestion")
dbutils.widgets.text("preco_bq_tb_usd", "[PREENCHER]", "Preço USD por TB lido no BigQuery — PREENCHER")
dbutils.widgets.text("preco_sfn_1k_transicoes_usd", "[PREENCHER]", "Preço USD por 1.000 transições Step Functions — PREENCHER")
dbutils.widgets.text("execucoes_mes", "30", "Execuções do pipeline por mês (ex.: diária = 30)")
dbutils.widgets.text("fonte_precos", "[PREENCHER: URL e data da cotação]", "Fonte/data dos preços")

def _num(nome):
    v = dbutils.widgets.get(nome).strip()
    try:
        return float(v)
    except ValueError:
        return None   # placeholder não preenchido -> custo fica NULO (nunca zero)

PRECO_DBU, DBU_H, PRECO_BQ, PRECO_SFN, EXEC_MES, DBU_EXEC = (_num(n) for n in
    ["preco_dbu_usd", "dbu_por_hora", "preco_bq_tb_usd", "preco_sfn_1k_transicoes_usd", "execucoes_mes", "dbu_por_execucao"])

In [0]:
# Quantidades MEDIDAS por execução (média das últimas execuções bem-sucedidas), sem dupla contagem:
# pipeline_runs já contém uma linha por (run_id, entity); somamos por run_id e depois tiramos a média.
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.monitoring")
horas_por_execucao = spark.sql("""
    SELECT AVG(h) AS horas FROM (
        SELECT run_id, SUM(duration_seconds)/3600.0 AS h
        FROM workspace.monitoring.pipeline_runs
        WHERE status = 'SUCESSO' GROUP BY run_id)""").first()["horas"]
print("Horas de compute por execução (medido, média por run_id):", horas_por_execucao)
# Observação: pipeline_runs registra uma linha por notebook; run_id é por notebook, então esta média é por NOTEBOOK.
# Para o total por execução do pipeline, multiplique pelo número de notebooks do Job (ver README) ou use system.billing.usage.

Horas de compute por execução (medido, média por run_id): 0.01073204418562553


In [0]:
from pyspark.sql import Row
from datetime import datetime, timezone

n_notebooks = spark.sql("""SELECT COUNT(DISTINCT entity) n FROM workspace.monitoring.pipeline_runs
                           WHERE status = 'SUCESSO'""").first()["n"]
horas_pipeline = None if horas_por_execucao is None else horas_por_execucao * n_notebooks
mb_bq = spark.sql("SHOW TABLES IN workspace.monitoring LIKE 'finops_ingestion'").count()
tb_bq_por_exec = None
if mb_bq:
    try:
        # uma execução completa = o ÚLTIMO registro de cada entidade (não o histórico inteiro)
        tb_bq_por_exec = spark.sql("""
            SELECT SUM(bytes_billed)/1e12 AS tb FROM (
              SELECT bytes_billed, ROW_NUMBER() OVER (PARTITION BY entity ORDER BY recorded_at DESC) rn
              FROM workspace.monitoring.finops_ingestion) WHERE rn = 1""").first()["tb"]
    except Exception as exc:   # coluna/tabela diferente do esperado: custo BigQuery fica NULO, não zero
        print("finops_ingestion sem bytes_billed utilizável:", str(exc)[:150])

_bq = _num("bq_gib_por_execucao")
if _bq is not None:   # medição manual (dry run) tem prioridade quando finops_ingestion ainda não tem bytes
    tb_bq_por_exec = _bq / 1024.0   # GiB -> TiB (unidade de cobrança do BigQuery)

def custo(qtd, preco):
    return None if (qtd is None or preco is None) else float(qtd) * float(preco)

linhas = [
    ("Databricks serverless (jobs)", "DBU-hora",
     (DBU_EXEC * EXEC_MES if (DBU_EXEC is not None and EXEC_MES is not None)   # medido na cobrança (preferido)
      else None if (horas_pipeline is None or DBU_H is None or EXEC_MES is None)
      else horas_pipeline * DBU_H * EXEC_MES), PRECO_DBU),
    ("BigQuery (ingestão on-demand)", "TB lido",
     None if (tb_bq_por_exec is None or EXEC_MES is None) else tb_bq_por_exec * EXEC_MES, PRECO_BQ),
    ("AWS Step Functions (Standard)", "1.000 transições",
     None if EXEC_MES is None else (EXEC_MES * 142) / 1000.0, PRECO_SFN),  # 142 transições/execução MEDIDAS em 26/10/2026 (133 nas 7 execuções filhas + 9 do pai)
]
agora = datetime.now(timezone.utc).replace(tzinfo=None)
rows = [Row(servico=s, unidade=u, quantidade_mensal=q, preco_unitario_usd=p, custo_mensal_usd=custo(q, p),
            premissa_precos=dbutils.widgets.get("fonte_precos"), calculado_em=agora) for s, u, q, p in linhas]
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
_schema = StructType([   # schema explícito: colunas 100% nulas (preço/quantidade não preenchidos) não podem ser inferidas
    StructField("servico", StringType()), StructField("unidade", StringType()),
    StructField("quantidade_mensal", DoubleType()), StructField("preco_unitario_usd", DoubleType()),
    StructField("custo_mensal_usd", DoubleType()), StructField("premissa_precos", StringType()),
    StructField("calculado_em", TimestampType())])
df = spark.createDataFrame(
    [(r.servico, r.unidade, r.quantidade_mensal, r.preco_unitario_usd, r.custo_mensal_usd,
      r.premissa_precos, r.calculado_em) for r in rows], _schema)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.monitoring.finops_projecao_mensal")
display(spark.table("workspace.monitoring.finops_projecao_mensal"))
total = sum(r.custo_mensal_usd for r in rows if r.custo_mensal_usd is not None)
print(f"Total mensal projetado (apenas linhas com preço informado): US$ {total:,.2f}")
print("Linhas com custo NULO = preço ou quantidade ainda não preenchidos/medidos. Não são custo zero.")

servico,unidade,quantidade_mensal,preco_unitario_usd,custo_mensal_usd,premissa_precos,calculado_em
Databricks serverless (jobs),DBU-hora,109.08,0.35,38.178,Databricks: system.billing.list_prices; AWS e Google: tabelas públicas de preço,2026-10-28T00:35:44.212Z
BigQuery (ingestão on-demand),TB lido,0.00744140625,6.25,0.0465087890625,Databricks: system.billing.list_prices; AWS e Google: tabelas públicas de preço,2026-10-28T00:35:44.212Z
AWS Step Functions (Standard),1.000 transições,4.26,0.025,0.1065,Databricks: system.billing.list_prices; AWS e Google: tabelas públicas de preço,2026-10-28T00:35:44.212Z


Total mensal projetado (apenas linhas com preço informado): US$ 38.33
Linhas com custo NULO = preço ou quantidade ainda não preenchidos/medidos. Não são custo zero.
